In [1]:
from deck import full_euchre_deck
from dealer import Dealer
from n_game_sim import n_game_sim
from n_play_round import round1, n_play_round, next_round
from n_branches import array_set_difference, n_find_winner, array_set_difference
from tree_search import find_best_opener, n_tree_sim, four_trick_sim, trick_played #, r2_best_response
import numpy as np
from numba import njit

In [2]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands5

array([[[ 11,   0],
        [  0, 100],
        [  0,  -9],
        [  9,   0],
        [-12,   0]],

       [[-14,   0],
        [ 12,   0],
        [  0,  90],
        [ 14,   0],
        [ -9,   0]],

       [[  0, 135],
        [ 10,   0],
        [ 13,   0],
        [  0, -12],
        [  0, -10]],

       [[-11,   0],
        [-13,   0],
        [  0, 120],
        [  0, 140],
        [  0, 130]]])

In [3]:
best_opener = find_best_opener(hands=hands5, player=0, lead=0, tricks=5, sim_func=n_tree_sim)

[0.05592573 0.07829181 0.05641625 0.06070703 0.05912556]


In [4]:
r2_leads, r2_hands = round1(hands_dealt=hands5, chosen_card=best_opener, leader=0)

In [5]:
# find the best response by iterating through each possible trick from the perspective of the opponent
# need to make this into a function as well

eval_position=0

r1_response_res = np.zeros(r2_leads.shape, dtype=np.float64)

for w in range(r2_leads.shape[0]):

    r3_leads, r3_hands, r2_score = next_round(
        current_hands=np.array([r2_hands[w]]),
        leads=np.array([r2_leads[w]]),
        game_round=2,
        game_score=np.array([r2_leads[w]]).reshape(-1, 1),
    )
    r4_leads, r4_hands, r3_score = next_round(
        current_hands=r3_hands, leads=r3_leads, game_round=3, game_score=r2_score
    )
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    if eval_position % 2 == 0:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2)<3
    else:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2)>=3

    r1_response_res[w] = np.mean(meta_results)
best_response = np.argmin(r1_response_res)
print(np.argmin(r1_response_res))

1


In [6]:
trick_played(hands5, r2_hands[best_response])

array([[  0, 100],
       [  0,  90],
       [  0, 135],
       [  0, 140]])

In [7]:
r2_leads[best_response]

np.int64(3)

In [8]:
r1_winner = r2_leads[best_response]

In [9]:
r2_optimal =  r2_hands[best_response]

In [10]:
r2_optimal

array([[[ 11,   0],
        [  0,  -9],
        [  9,   0],
        [-12,   0]],

       [[-14,   0],
        [ 12,   0],
        [ 14,   0],
        [ -9,   0]],

       [[ 10,   0],
        [ 13,   0],
        [  0, -12],
        [  0, -10]],

       [[-11,   0],
        [-13,   0],
        [  0, 120],
        [  0, 130]]])

In [11]:
r2_best_opener = find_best_opener(hands=r2_optimal, player=0, lead=r1_winner, tricks=4, sim_func=four_trick_sim)

[0. 0. 0. 0.]


In [12]:
r1_winner

np.int64(3)

In [13]:
r3_leads, r3_hands = round1(hands_dealt=r2_optimal, chosen_card=r2_best_opener, leader=r1_winner)

In [14]:
eval_position=0

r2_response_res = np.zeros(r3_leads.shape, dtype=np.float64)

for w in range(r3_leads.shape[0]):

    r4_leads, r4_hands, r3_score = next_round(
        current_hands=np.array([r3_hands[w]]),
        leads=np.array([r3_leads[w]]),
        game_round=3,
        game_score=np.array([r3_leads[w]]).reshape(-1, 1),
    )
    
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    if eval_position % 2 == 0:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2 + (r1_winner%2))<3
    else:
        for i in range(len(results)):
            meta_results[i] = np.sum(results[i]%2 + (r1_winner%2))>=3

    r2_response_res[w] = np.mean(meta_results)
best_response = np.argmin(r2_response_res)
print(np.argmin(r2_response_res))

0


In [15]:
trick_played(r2_optimal, r3_hands[best_response])

array([[-12,   0],
       [-14,   0],
       [ 10,   0],
       [-11,   0]])

In [16]:
r2_winner = r3_leads[best_response]

In [17]:
r3_optimal =  r3_hands[best_response]

In [18]:
r3_optimal

array([[[ 11,   0],
        [  0,  -9],
        [  9,   0]],

       [[ 12,   0],
        [ 14,   0],
        [ -9,   0]],

       [[ 13,   0],
        [  0, -12],
        [  0, -10]],

       [[-13,   0],
        [  0, 120],
        [  0, 130]]])

In [19]:
def three_trick_sim(
    game_hand: np.ndarray, eval_position: int, r1_chosen_card: np.ndarray, lead: int, rd1_winner: int
):
    """
    Simulates a full game using the provided hands and evaluates the outcome based on
    a specified player's position. The function runs five rounds of play, tracking the
    results and calculating the final score based on the player's performance. It then
    returns the expected value of the hand based on the scored outcomes.

    Arguments:
        game_hand (numpy.ndarray): A 3D array representing the cards dealt to each player.
        eval_position (int): The index of the player whose cards are being evaluated.

    Returns:
        float: The mean performance evaluation for the specified player, indicating the
        proportion of games where the player's performance exceeds a threshold.
    """
    r3_leads, r3_hands = round1(
        hands_dealt=game_hand, chosen_card=r1_chosen_card, leader=lead
    )

    r4_leads, r4_hands, r3_score = next_round(
        current_hands=r2_hands,
        leads=r2_leads,
        game_round=3,
        game_score=r2_leads.reshape(-1, 1),
    )

    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=5, game_score=r3_score
    )

    results = r4_score.reshape(r4_score.shape[0], 5)
    for i in range(results.shape[0]):
        results[i][1] = lead

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    if eval_position % 2 == 0:
        for i in range(len(results)):
            meta_results[i] = (np.sum(results[i] % 2) + (lead % 2) + (rd1_winner%2)) < 3
    else:
        for i in range(len(results)):
            meta_results[i] = (np.sum(results[i] % 2) + (lead % 2)+ (rd1_winner%2)) >= 3

    return np.mean(meta_results)

In [20]:
from typing import Callable

def rd_3_find_best_opener(
    hands: np.ndarray, player: int, lead: int, tricks: int, rd1_winner: int, sim_func: Callable
):
    winning_chances = np.zeros(tricks)
    for i in range(tricks):
        winning_chances[i] = sim_func(
            game_hand=hands, eval_position=player, r1_chosen_card=i, lead=lead, rd1_winner=rd1_winner
        )

    print(winning_chances)
    return np.argmax(winning_chances)

In [21]:
r3_best_opener = rd_3_find_best_opener(hands=r3_optimal, player=0, lead=r2_winner, tricks=3, rd1_winner=r1_winner, sim_func=three_trick_sim)

[0. 0. 0.]


In [22]:
r2_winner

np.int64(1)

In [ ]:
# some general notes:
# I need to make sure the response logic is sound. Basically, I should probably drop the whole, if else logic when calculating total scores. 
# I need to just establish some thing like, for positions 0 and 2, values close to 0 are good. For positions 1 and 3, values close to 1 are good. 
# That way it would be easy to confirm best responses. 
# Should probably move the functions in here to tree search and figure out the best way to refactor without having to repeat code so much